# 05b — Parallel Tempering Results — ANALYSIS

**Companion to**: `05b_sampling_tempered_data.ipynb` (sampling)

This notebook loads saved tempered NUTS samples and runs all diagnostics:
1. Swap acceptance rates (per temperature pair)
2. Log-probability trace plots
3. ACF heatmap in Hessian eigenbasis
4. ESS comparison vs H1 baseline
5. Spectral variance diagnostics
6. Corner plot (degenerate vs non-degenerate)
7. Mass matrix vs Hessian
8. **H3-1 iterative convergence** — per-round ACF/ESS/spectral/sample-quality
   plots that mirror H1-1, so the two methodologies can be directly compared.

**Usage**: Set `RUN_NAME` below to load a specific run.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Resolve project root
cwd = Path.cwd().resolve()
project_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "packages" / "pytorch_models" / "markov_transformer.py").exists()
    ),
    None,
)
if project_root is None:
    alt_root = cwd / "projects" / "markov-chain-learning"
    if (alt_root / "packages" / "pytorch_models" / "markov_transformer.py").exists():
        project_root = alt_root
if project_root is None:
    raise RuntimeError("Could not locate markov-chain-learning project root")

packages_dir = project_root / "packages"
if str(packages_dir) not in sys.path:
    sys.path.insert(0, str(packages_dir))

from pytorch_models import MarkovTransformer

DATA_DIR = project_root / "experiments" / "single-chain" / "data"
print(f"Project root: {project_root}")
print(f"Data dir: {DATA_DIR}")

In [ ]:
# ── Parameters ──
RUN_NAME: str = "default"  # which tempered run to analyze
SIGMA_PRIOR: float = 10.0  # must match training

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD CHECKPOINT
# ══════════════════════════════════════════════════════════════════════════════
RESULTS_DIR = DATA_DIR / "results_tempered" / RUN_NAME

saved = torch.load(RESULTS_DIR / "nuts_samples.pt", weights_only=False)

# Unpack
run_params = saved["params"]
BETAS = run_params["betas"]
N_CHAINS = len(BETAS)
N_SAMPLES_PT = run_params["n_samples"]
N_WARMUP_PT = run_params["n_warmup"]
elapsed = run_params.get("elapsed_seconds", 0)
mle_param = saved["mle_param"]

# Cold chain (β=1.0) samples
cold_samples = saved["chains"][0]["parameters"].float()
cold_logp = saved["chains"][0].get("log_probabilities", None)

# All chains' data
all_chains = saved["chains"]

# Swap diagnostics
swap_diag = saved["swap_diagnostics"]
swap_rate = swap_diag["swap_acceptance_rate"]
n_proposed = swap_diag["n_swaps_proposed"]
n_accepted = swap_diag["n_swaps_accepted"]
per_pair_rates = swap_diag.get("per_pair_rates", {})
swap_history = saved.get("swap_history", [])

print(f"✓ Loaded run '{RUN_NAME}' from {RESULTS_DIR}")
print(f"  {N_CHAINS} chains, β = {BETAS}")
print(f"  {cold_samples.shape[0]} samples × d={cold_samples.shape[1]}")
print(f"  Elapsed: {elapsed / 60:.1f} min")
print(f"  Swap rate: {swap_rate:.3f} ({n_accepted}/{n_proposed})")
if cold_logp is not None:
    print(f"  Log-prob available: {len(cold_logp)} values")

In [ ]:
# ── (Model/sequences load no longer needed — Hessian comes from hessian.pt) ──
# DGP_REGIME / VOCAB_SIZE etc. are inside the saved nuts_samples payload (see next cell).


## 1. Swap Acceptance Diagnostics

In [ ]:
# ── Swap diagnostics ──
print(f"Overall swap acceptance rate: {swap_rate:.3f}")
print(f"  Proposed: {n_proposed}, Accepted: {n_accepted}")

n_pairs = len(BETAS) - 1
print(f"\nTemperature pairs ({n_pairs} adjacent):")
for i in range(n_pairs):
    print(
        f"  β={BETAS[i]:.2f} ↔ β={BETAS[i + 1]:.2f}  (Δβ = {BETAS[i] - BETAS[i + 1]:.2f})"
    )

# Get per-pair rates for the bar chart
bar_labels = [f"{BETAS[i]:.2f}↔{BETAS[i + 1]:.2f}" for i in range(n_pairs)]
if per_pair_rates:
    bar_values = []
    for i in range(n_pairs):
        found = False
        for key in per_pair_rates:
            if f"{BETAS[i]}" in key and f"{BETAS[i + 1]}" in key:
                bar_values.append(per_pair_rates[key])
                found = True
                break
        if not found:
            bar_values.append(swap_rate)
else:
    bar_values = [swap_rate] * n_pairs

fig, ax = plt.subplots(figsize=(12, 5))
colors = [
    "green" if v >= 0.2 else ("orange" if v >= 0.1 else "red") for v in bar_values
]
ax.bar(bar_labels, bar_values, color=colors, alpha=0.8, edgecolor="k", linewidth=0.5)
ax.axhline(0.2, color="red", ls="--", lw=1.5, label="Minimum viable (20%)")
ax.axhline(0.5, color="green", ls=":", lw=1.5, label="Ideal target (50%)")
ax.set_xlabel("Temperature pair", fontsize=12)
ax.set_ylabel("Swap acceptance rate", fontsize=12)
ax.set_title(
    "Replica Exchange Swap Acceptance (per pair)", fontsize=13, fontweight="bold"
)
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("\nPer-pair rates:")
for label, val in zip(bar_labels, bar_values):
    status = "✓" if val >= 0.2 else ("⚠" if val >= 0.1 else "✗")
    print(f"  {status} {label}: {val:.3f}")

## 2. Log-Probability Trace Plots

In [ ]:
# ── Trace plot: log p(θ|D) per chain ──
fig, axes = plt.subplots(N_CHAINS, 1, figsize=(14, 2.5 * N_CHAINS), sharex=True)
if N_CHAINS == 1:
    axes = [axes]

for ci, chain_data in enumerate(all_chains):
    ax = axes[ci]
    logp = chain_data.get("log_probabilities", None)
    if logp is not None:
        if isinstance(logp, torch.Tensor):
            logp = logp.numpy()
        else:
            logp = np.array(logp)
        ax.plot(logp, lw=0.5, color="teal", alpha=0.8)
        ax.set_ylabel("log p(θ|D)", fontsize=9)
    else:
        ax.text(
            0.5,
            0.5,
            "log_probabilities not saved",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=10,
            color="gray",
        )
    ax.set_title(
        f"Chain {ci} (β={chain_data['beta']:.2f}, accept={chain_data['acceptance_rate']:.3f})",
        fontsize=10,
        loc="left",
    )
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample index")
fig.suptitle("Log-probability traces (all chains)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Hessian Eigenbasis & ACF Heatmap

In [ ]:
# ── Load Hessian (computed once in 04_training.ipynb) ──
hess = torch.load(DATA_DIR / "hessian.pt", weights_only=True)
H_mle = hess["H_mle"].float()
eigvals_h, eigvecs_h = torch.linalg.eigh(H_mle)
eigval_threshold = eigvals_h.abs().max().item() * 1e-3
n_nd = int((eigvals_h.abs() > eigval_threshold).sum())
n_degen = int(H_mle.shape[0]) - n_nd
print(f"Hessian: λ ∈ [{eigvals_h[0]:.4e}, {eigvals_h[-1]:.4e}]")
print(f"  {n_nd} non-degenerate, {n_degen} degenerate directions")


In [ ]:
# ── ACF per Hessian eigendirection ──
MAX_LAG = min(2000, N_SAMPLES_PT // 4)


def compute_acf_eigenbasis(samples_tensor, V, max_lag):
    """Project samples into eigenbasis, compute ACF per direction via FFT."""
    z = samples_tensor @ V
    d = z.shape[1]
    acf = torch.zeros(d, max_lag)
    for j in range(d):
        x = z[:, j]
        x = x - x.mean()
        var = x.var()
        if var < 1e-20:
            continue
        n = x.shape[0]
        padded = torch.zeros(2 * n)
        padded[:n] = x
        ft = torch.fft.rfft(padded)
        acov = torch.fft.irfft(ft * ft.conj())[:n] / n
        acf[j, :max_lag] = acov[:max_lag] / acov[0].clamp(min=1e-20)
    return acf


acf_sorted = compute_acf_eigenbasis(cold_samples, V, MAX_LAG)[sort_idx]

# ── ACF heatmap ──
fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(
    acf_sorted.numpy(),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-0.3,
    vmax=1.0,
    interpolation="nearest",
    origin="upper",
)
ax.set_xlabel("Lag τ", fontsize=12)
ax.set_ylabel("Eigendirection (sorted by λ, largest at top)", fontsize=12)
ax.set_title(
    f"ACF — Tempered NUTS cold chain (β=1.0, N={N_SAMPLES_PT})\n"
    f"d={mle_param.shape[0]}, warmup={N_WARMUP_PT}",
    fontsize=13,
    fontweight="bold",
)
if 0 < n_nd < cold_samples.shape[1]:
    ax.axhline(
        n_nd - 0.5,
        color="lime",
        lw=2,
        ls="--",
        label=f"degen boundary (top {n_nd} non-degen)",
    )
    ax.legend(loc="upper right", fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8, label="ACF")
plt.tight_layout()
plt.show()

# Summary
acf_at_50 = acf_sorted[:, min(50, MAX_LAG - 1)]
acf50_nd = acf_sorted[:n_nd, min(50, MAX_LAG - 1)]
acf50_dg = acf_sorted[n_nd:, min(50, MAX_LAG - 1)]
print(f"\nACF@50 all:      median={acf_at_50.median():.3f}")
print(f"ACF@50 non-degen: median={acf50_nd.median():.3f}")
print(f"ACF@50 degenerate: median={acf50_dg.median():.3f}")

## 4. ESS Comparison vs Baseline

In [ ]:
# ── ESS ──
def compute_ess(acf_row):
    total = 0.0
    for k in range(acf_row.shape[0]):
        if acf_row[k] < 0:
            break
        total += acf_row[k]
    tau = 1 + 2 * (total - 1)
    return N_SAMPLES_PT / max(tau, 1.0)


ess_tempered = torch.tensor(
    [compute_ess(acf_sorted[i]) for i in range(acf_sorted.shape[0])]
)
ess_nd_t = ess_tempered[:n_nd]
ess_dg_t = ess_tempered[n_nd:]

print(f"ESS (tempered cold chain, N={N_SAMPLES_PT}):")
print(
    f"  All:       min={ess_tempered.min():.1f}, median={ess_tempered.median():.1f}, max={ess_tempered.max():.1f}"
)
print(f"  Non-degen: median={ess_nd_t.median():.1f}")
print(f"  Degenerate: median={ess_dg_t.median():.1f}")

# Load H1 baseline
baseline_dir = DATA_DIR / "results_no_hessian"
baseline_tensors = torch.load(
    baseline_dir / "diagnostics_tensors.pt", weights_only=True
)
ess_baseline = baseline_tensors["ess_per_dir"]

# Improvement factor (normalize to per-sample rate)
ess_rate_tempered = ess_tempered / N_SAMPLES_PT
ess_rate_baseline = ess_baseline / 10000
improvement = ess_rate_tempered / ess_rate_baseline.clamp(min=1e-6)
print(
    f"\nESS improvement over H1: median={improvement.median():.1f}×, min={improvement.min():.2f}×, max={improvement.max():.1f}×"
)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].bar(
    range(len(ess_tempered)),
    ess_tempered.numpy(),
    width=1.0,
    color="darkorange",
    alpha=0.7,
    label=f"Tempered (N={N_SAMPLES_PT})",
)
if 0 < n_nd < len(ess_tempered):
    axes[0].axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
axes[0].set_ylabel("ESS")
axes[0].set_title("H3: Tempered NUTS (cold chain)", fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].bar(
    range(len(ess_baseline)),
    ess_baseline.numpy(),
    width=1.0,
    color="steelblue",
    alpha=0.7,
    label="H1 baseline (N=10000)",
)
if 0 < n_nd < len(ess_baseline):
    axes[1].axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
axes[1].set_xlabel("Eigendirection (sorted by λ)")
axes[1].set_ylabel("ESS")
axes[1].set_title("H1: Single-chain NUTS (baseline)", fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis="y")

fig.suptitle("ESS Comparison: Tempered vs Untempered", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Spectral Variance Diagnostics

In [ ]:
# ── Sample variance in eigenbasis ──
z_cold = cold_samples @ V
var_sorted = (z_cold - z_cold.mean(dim=0, keepdim=True)).var(dim=0)[sort_idx]
prior_var = SIGMA_PRIOR**2
rank_idx = torch.arange(1, len(var_sorted) + 1)

# H1 baseline variance
h1_data = torch.load(DATA_DIR / "results_no_hessian" / BASELINE_RUN / "nuts_samples_no_hessian.pt", weights_only=False)
h1_samples = h1_data["chains"][0]["parameters"].float()
z_h1 = h1_samples @ V
var_h1 = (z_h1 - z_h1.mean(dim=0, keepdim=True)).var(dim=0)[sort_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.semilogy(
    rank_idx.numpy(),
    var_sorted.numpy(),
    "o",
    ms=2,
    color="teal",
    alpha=0.6,
    label="Tempered cold chain",
)
ax1.axhline(prior_var, color="red", ls="--", lw=1.5, label=f"Prior var = {prior_var}")
if 0 < n_nd < len(var_sorted):
    ax1.axvline(n_nd, color="lime", lw=2, ls="--", alpha=0.7, label="degen boundary")
ax1.set_xlabel("Eigendirection rank")
ax1.set_ylabel("Variance (log)")
ax1.set_title("Sample variance per eigendirection", fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.semilogy(
    rank_idx.numpy(),
    var_sorted.numpy(),
    "o",
    ms=2,
    color="darkorange",
    alpha=0.6,
    label="Tempered",
)
ax2.semilogy(
    rank_idx.numpy(),
    var_h1.numpy(),
    "o",
    ms=2,
    color="steelblue",
    alpha=0.4,
    label="H1 baseline",
)
ax2.axhline(prior_var, color="red", ls="--", lw=1, alpha=0.5)
if 0 < n_nd < len(var_sorted):
    ax2.axvline(n_nd, color="lime", lw=2, ls="--", alpha=0.7)
ax2.set_xlabel("Eigendirection rank")
ax2.set_ylabel("Variance (log)")
ax2.set_title("Tempered vs H1", fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle(
    "Spectral diagnostics: cold-chain exploration", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

var_ratio = var_sorted / var_h1.clamp(min=1e-10)
print(f"Variance ratio (tempered / H1):")
print(f"  Non-degen: median = {var_ratio[:n_nd].median():.2f}×")
print(f"  Degenerate: median = {var_ratio[n_nd:].median():.2f}×")

## 6. Corner Plot: Degenerate vs Non-Degenerate

In [ ]:
# ── Corner plot ──
z_cold_sorted = cold_samples @ V[:, sort_idx]

# Select 3 non-degenerate + 3 degenerate eigendirections
nd_indices = [0, n_nd // 2, n_nd - 1]
dg_indices = [n_nd, n_nd + n_degen // 2, n_nd + n_degen - 1]
selected_indices = nd_indices + dg_indices
n_sel = len(selected_indices)

labels = []
for idx in selected_indices:
    lam = eigvals_sorted[idx].item()
    tag = "ND" if idx < n_nd else "DG"
    rank = idx if idx < n_nd else idx - n_nd
    labels.append(f"{tag} {rank}\nλ={lam:.1e}")

z_sel = z_cold_sorted[:, selected_indices].numpy()

fig, axes = plt.subplots(n_sel, n_sel, figsize=(16, 16))

for i in range(n_sel):
    for j in range(n_sel):
        ax = axes[i, j]
        if j > i:
            ax.axis("off")
            continue
        if i == j:
            ax.hist(
                z_sel[:, i],
                bins=40,
                density=True,
                color="teal",
                alpha=0.7,
                edgecolor="none",
            )
            ax.set_yticks([])
            if i == n_sel - 1:
                ax.set_xlabel(labels[j], fontsize=8)
        else:
            ax.scatter(
                z_sel[:, j],
                z_sel[:, i],
                s=1,
                alpha=0.3,
                color="darkorange",
                rasterized=True,
            )
            try:
                from scipy.stats import gaussian_kde

                xy = np.vstack([z_sel[:, j], z_sel[:, i]])
                if xy[0].std() > 1e-10 and xy[1].std() > 1e-10:
                    kde = gaussian_kde(xy)
                    xmin, xmax = xy[0].min(), xy[0].max()
                    ymin, ymax = xy[1].min(), xy[1].max()
                    pad_x, pad_y = (xmax - xmin) * 0.1, (ymax - ymin) * 0.1
                    xx, yy = np.mgrid[
                        xmin - pad_x : xmax + pad_x : 50j,
                        ymin - pad_y : ymax + pad_y : 50j,
                    ]
                    zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
                    ax.contour(
                        xx, yy, zz, levels=5, colors="k", linewidths=0.5, alpha=0.6
                    )
            except (ImportError, np.linalg.LinAlgError):
                pass
            if i == n_sel - 1:
                ax.set_xlabel(labels[j], fontsize=8)
            if j == 0:
                ax.set_ylabel(labels[i], fontsize=8)
        if i < n_sel - 1:
            ax.set_xticklabels([])
        if j > 0 and i != j:
            ax.set_yticklabels([])

fig.suptitle(
    f"Corner plot: 3 Non-Degenerate + 3 Degenerate eigendirections\n"
    f"Tempered NUTS cold chain (β=1.0, N={N_SAMPLES_PT})",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("\nSelected eigendirections:")
for k, idx in enumerate(selected_indices):
    lam = eigvals_sorted[idx].item()
    var = z_sel[:, k].var()
    tag = "NON-DEGEN" if idx < n_nd else "DEGENERATE"
    print(f"  [{tag}] rank {idx}: λ={lam:.3e}, sample_var={var:.3e}")

## 7. Mass Matrix vs Hessian

In [ ]:
# ── Compare adapted mass matrix to Hessian ──
cold_diag = all_chains[0]["diagnostics"]
M_cold = cold_diag.get("adapted_mass_matrix", None)

if M_cold is not None:
    if isinstance(M_cold, torch.Tensor):
        M_cold = M_cold.float()
    else:
        M_cold = torch.tensor(M_cold).float()

    print(f"Mass matrix shape: {M_cold.shape}")
    print(f"  type: {'diagonal' if M_cold.dim() == 1 else 'full'}")

    if M_cold.dim() == 1:
        # Diagonal mass: compare to Hessian diagonal in eigenbasis
        # The mass matrix lives in parameter space, project to eigenbasis
        # For diagonal M in param space: M_eig = V^T diag(M) V
        # Just compare spectra
        M_eig_diag = (V.T @ torch.diag(M_cold) @ V).diag()[sort_idx]

        fig, ax = plt.subplots(figsize=(14, 5))
        ax.semilogy(
            rank_idx.numpy(),
            eigvals_sorted.abs().numpy(),
            "o",
            ms=2,
            alpha=0.5,
            color="steelblue",
            label="|Hessian eigenvalues|",
        )
        ax.semilogy(
            rank_idx.numpy(),
            M_eig_diag.numpy(),
            "o",
            ms=2,
            alpha=0.5,
            color="darkorange",
            label="Mass matrix (eigenbasis diag)",
        )
        if 0 < n_nd < len(eigvals_sorted):
            ax.axvline(
                n_nd, color="lime", lw=2, ls="--", alpha=0.7, label="degen boundary"
            )
        ax.set_xlabel("Eigendirection rank")
        ax.set_ylabel("Value (log)")
        ax.set_title("Adapted Mass Matrix vs Hessian Eigenvalues", fontweight="bold")
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    elif M_cold.dim() == 2:
        # Full mass matrix: compute its eigenvalues
        M_eigvals = torch.linalg.eigvalsh(M_cold)
        M_eigvals_sorted = M_eigvals.sort(descending=True).values

        fig, ax = plt.subplots(figsize=(14, 5))
        ax.semilogy(
            rank_idx.numpy(),
            eigvals_sorted.abs().numpy(),
            "o",
            ms=2,
            alpha=0.5,
            color="steelblue",
            label="|Hessian eigenvalues|",
        )
        ax.semilogy(
            rank_idx.numpy(),
            M_eigvals_sorted.numpy(),
            "o",
            ms=2,
            alpha=0.5,
            color="darkorange",
            label="Mass matrix eigenvalues",
        )
        if 0 < n_nd < len(eigvals_sorted):
            ax.axvline(
                n_nd, color="lime", lw=2, ls="--", alpha=0.7, label="degen boundary"
            )
        ax.set_xlabel("Rank")
        ax.set_ylabel("Eigenvalue (log)")
        ax.set_title("Mass Matrix Spectrum vs Hessian", fontweight="bold")
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No adapted mass matrix saved (identity mass was used)")

# Per-chain summary
print("\nPer-chain diagnostics:")
for ci, ch in enumerate(all_chains):
    d = ch["diagnostics"]
    eps = d.get("adapted_step_size", d.get("step_size", "?"))
    n_div = d.get("n_divergences", 0)
    depth = d.get("mean_tree_depth", "?")
    print(
        f"  Chain {ci} (β={ch['beta']:.2f}): ε={eps:.4e}, mean_depth={depth}, divergences={n_div}"
    )

## 8. Save Results & Verdict

In [ ]:
# ── Save all figures and numerical results ──
acf_at_10 = acf_sorted[:, min(10, MAX_LAG - 1)]
acf_at_200 = acf_sorted[:, min(200, MAX_LAG - 1)]
acf50_median = float(acf_at_50.median())

results = {
    "experiment": "04_tempered",
    "hypothesis": "H3: parallel tempering breaks singular mixing barrier",
    "params": run_params,
    "swap_diagnostics": {
        "n_swaps_proposed": n_proposed,
        "n_swaps_accepted": n_accepted,
        "swap_acceptance_rate": float(swap_rate),
        "per_pair_rates": per_pair_rates,
    },
    "acf": {
        "lag_10_median": float(acf_at_10.median()),
        "lag_50_median": acf50_median,
        "lag_50_nondegen": float(acf50_nd.median()),
        "lag_50_degen": float(acf50_dg.median()),
        "lag_200_median": float(acf_at_200.median()),
    },
    "ess": {
        "all_min": float(ess_tempered.min()),
        "all_median": float(ess_tempered.median()),
        "all_max": float(ess_tempered.max()),
        "nondegen_median": float(ess_nd_t.median()),
        "degen_median": float(ess_dg_t.median()),
    },
    "ess_improvement": {
        "median_factor": float(improvement.median()),
        "min_factor": float(improvement.min()),
        "max_factor": float(improvement.max()),
    },
    "verdict": "confirmed"
    if acf50_median < 0.3
    else ("inconclusive" if acf50_median < 0.5 else "falsified"),
}

with open(RESULTS_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

torch.save(
    {
        "acf_sorted": acf_sorted,
        "ess_tempered": ess_tempered,
        "ess_baseline": ess_baseline,
        "var_sorted": var_sorted,
        "var_h1": var_h1,
        "eigvals_sorted": eigvals_sorted,
        "n_nd": n_nd,
        "n_degen": n_degen,
        "improvement": improvement,
    },
    RESULTS_DIR / "diagnostics_tensors.pt",
)

print(f"✓ Saved to {RESULTS_DIR}/")
print(f"\n{'=' * 70}")
print(f"  H3 VERDICT: {results['verdict'].upper()}")
print(f"{'=' * 70}")
print(f"  ACF@50 median = {acf50_median:.3f}  (< 0.3 confirmed, > 0.5 falsified)")
print(f"  ESS median = {float(ess_tempered.median()):.1f} / {N_SAMPLES_PT}")
print(f"  ESS improvement: {float(improvement.median()):.1f}×")
print(f"  Swap acceptance: {swap_rate:.3f}")
print(f"  Wall time: {elapsed / 60:.1f} min")

## 9. H3-1: Iterative Convergence (mirrors H1-1)

Load per-round checkpoints from `iterative/round_NN.pt`, recompute ACF/ESS
per round in the same Hessian eigenbasis, plot convergence side-by-side.

This is the direct analogue of H1-1's `convergence_summary.{pdf,png}` and
makes H3-1 vs H1-1 directly comparable.


In [ ]:
# ── Load H3-1 per-round artifacts ──
import json as _json

ITER_DIR = RESULTS_DIR / "iterative"
iter_round_paths = sorted(ITER_DIR.glob("round_*.pt")) if ITER_DIR.exists() else []

# Reuse the cold-chain ACF helper + eigenbasis already built earlier in this notebook
# (V, sort_idx, eigvals_sorted, n_nd, H_mle, eigvecs, eigvals, compute_acf_eigenbasis,
#  prior_var, SIGMA_PRIOR are all in scope).

# Compute H_trunc once (for spectral match diagnostic)
eigvals_h_trunc = eigvals.clamp(min=0)
H_trunc = eigvecs @ torch.diag(eigvals_h_trunc) @ eigvecs.T

iter_results = []
iter_acf_sorted = []  # for per-round heatmap strip if desired
iter_var_sorted = []  # spectral variance per round (cold chain)

if iter_round_paths:
    print(f"Found {len(iter_round_paths)} H3-1 round checkpoints")
    MAX_LAG_ITER = None

    for rp in iter_round_paths:
        rd = torch.load(rp, weights_only=False)
        samps_r = rd["samples"].float()  # cold-chain samples
        N_r = samps_r.shape[0]
        if MAX_LAG_ITER is None:
            MAX_LAG_ITER = min(500, max(N_r // 4, 2))

        acf_r = compute_acf_eigenbasis(samps_r, V, MAX_LAG_ITER)[sort_idx]
        acf50_r = acf_r[:, min(50, MAX_LAG_ITER - 1)]

        def _ess(row, N):
            tot = 0.0
            for k in range(row.shape[0]):
                if row[k] < 0:
                    break
                tot += float(row[k])
            tau = 1 + 2 * (tot - 1)
            return N / max(tau, 1.0)

        ess_r = torch.tensor([_ess(acf_r[i], N_r) for i in range(acf_r.shape[0])])

        # Sample variance per direction (cold chain)
        z_r = samps_r @ V
        var_r = (z_r - z_r.mean(dim=0, keepdim=True)).var(dim=0)[sort_idx]

        # Spectral: eig(H₊ · M_cold) for the round's cold-chain mass matrix
        M_cold_r = rd["per_chain_mass_matrix"][0]
        if M_cold_r is not None:
            M_cold_cpu = M_cold_r.detach().cpu().float()
            eigvals_mr, eigvecs_mr = torch.linalg.eigh(M_cold_cpu)
            M_sqrt_r = (
                eigvecs_mr
                @ torch.diag(eigvals_mr.clamp(min=1e-10).sqrt())
                @ eigvecs_mr.T
            )
            HtM_r = M_sqrt_r @ H_trunc @ M_sqrt_r
            eig_HtM_r = torch.sort(torch.linalg.eigvalsh(HtM_r), descending=True).values
            n_HtM_near1 = int(((eig_HtM_r - 1).abs() < 0.5).sum())
            sv = torch.linalg.svdvals(M_cold_cpu)
            cond_M = float(sv[0] / sv[-1]) if sv[-1] > 0 else float("inf")
        else:
            n_HtM_near1 = 0
            cond_M = float("nan")

        d_info = rd["diagnostics"]
        iter_results.append(
            {
                "round": int(rd["round"]),
                "step_size_cold": d_info["step_size_cold"],
                "cond_M_cold": cond_M,
                "acceptance_rate_cold": d_info["acceptance_rate_cold"],
                "n_divergences_cold": d_info["n_divergences_cold"],
                "swap_acceptance_rate": d_info.get("swap_acceptance_rate", 0.0),
                "acf50_median": float(acf50_r.median()),
                "acf50_nd_median": float(acf50_r[:n_nd].median()),
                "acf50_dg_median": float(acf50_r[n_nd:].median()) if n_nd < acf_r.shape[0] else float("nan"),
                "ess_median": float(ess_r.median()),
                "ess_nd_median": float(ess_r[:n_nd].median()),
                "ess_dg_median": float(ess_r[n_nd:].median()) if n_nd < acf_r.shape[0] else float("nan"),
                "n_HtM_near1": n_HtM_near1,
                "var_nd_median": float(var_r[:n_nd].median()),
                "var_dg_median": float(var_r[n_nd:].median()) if n_nd < var_r.shape[0] else float("nan"),
                "n_samples": N_r,
            }
        )
        iter_acf_sorted.append(acf_r)
        iter_var_sorted.append(var_r)

    print(f"Processed {len(iter_results)} rounds")
else:
    print("No H3-1 rounds found — skipping iterative analysis.")


In [ ]:
# ── H3-1 convergence plots (mirrors H1-1 convergence_summary) ──
if iter_results:
    rounds = [r["round"] for r in iter_results]
    acf50s = [r["acf50_median"] for r in iter_results]
    acf50_nds = [r["acf50_nd_median"] for r in iter_results]
    acf50_dgs = [r["acf50_dg_median"] for r in iter_results]
    ess_meds = [r["ess_median"] for r in iter_results]
    ess_nd_meds = [r["ess_nd_median"] for r in iter_results]
    ess_dg_meds = [r["ess_dg_median"] for r in iter_results]
    n_near1s = [r["n_HtM_near1"] for r in iter_results]
    conds = [r["cond_M_cold"] for r in iter_results]
    swap_rates = [r["swap_acceptance_rate"] for r in iter_results]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # ACF@50
    ax = axes[0, 0]
    ax.plot(rounds, acf50s, "o-", color="teal", lw=2, label="all")
    ax.plot(rounds, acf50_nds, "s--", color="darkorange", lw=1.5, label="non-degen")
    ax.plot(rounds, acf50_dgs, "^:", color="slateblue", lw=1.5, label="degen")
    ax.axhline(0.3, color="green", ls=":", lw=1, label="H3 confirmed (<0.3)")
    ax.axhline(0.5, color="red", ls=":", lw=1, label="H3 falsified (>0.5)")
    ax.set_xlabel("Round")
    ax.set_ylabel("ACF@50 median")
    ax.set_title("ACF@50 (cold chain) across rounds")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

    # ESS
    ax = axes[0, 1]
    ax.plot(rounds, ess_meds, "o-", color="steelblue", lw=2, label="all")
    ax.plot(rounds, ess_nd_meds, "s--", color="darkorange", lw=1.5, label="non-degen")
    ax.plot(rounds, ess_dg_meds, "^:", color="slateblue", lw=1.5, label="degen")
    ax.set_xlabel("Round")
    ax.set_ylabel("ESS median")
    ax.set_title("ESS (cold chain) across rounds")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Spectral match
    ax = axes[0, 2]
    ax.plot(rounds, n_near1s, "o-", color="purple", lw=2)
    ax.set_xlabel("Round")
    ax.set_ylabel("# modes with eig(H₊·M_cold) ∈ [0.5, 1.5]")
    ax.set_title("Spectral match (cold-chain M)")
    ax.grid(True, alpha=0.3)

    # Conditioning
    ax = axes[1, 0]
    ax.plot(rounds, conds, "o-", color="crimson", lw=2)
    ax.set_xlabel("Round")
    ax.set_ylabel("cond(M_cold)")
    ax.set_yscale("log")
    ax.set_title("Cold-chain mass matrix conditioning")
    ax.grid(True, alpha=0.3)

    # Swap acceptance rate per round (PT-specific, no analogue in H1-1)
    ax = axes[1, 1]
    ax.plot(rounds, swap_rates, "o-", color="darkgreen", lw=2)
    ax.axhline(0.2, color="red", ls="--", lw=1, alpha=0.7, label="min viable (0.2)")
    ax.axhline(0.5, color="green", ls=":", lw=1, alpha=0.7, label="ideal (0.5)")
    ax.set_xlabel("Round")
    ax.set_ylabel("Swap acceptance rate")
    ax.set_title("Replica-exchange swap rate across rounds")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

    # Sample quality: cold-chain variance vs prior
    ax = axes[1, 2]
    var_nd_meds = [r["var_nd_median"] for r in iter_results]
    var_dg_meds = [r["var_dg_median"] for r in iter_results]
    ax.plot(rounds, var_nd_meds, "s-", color="darkorange", lw=2, label="non-degen median")
    ax.plot(rounds, var_dg_meds, "^-", color="slateblue", lw=2, label="degen median")
    ax.axhline(prior_var, color="red", ls="--", lw=1, label=f"prior var = {prior_var}")
    ax.set_xlabel("Round")
    ax.set_ylabel("Sample variance (eigenbasis)")
    ax.set_yscale("log")
    ax.set_title("Sample-quality convergence (cold chain)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    fig.suptitle("H3-1: Iterative Tempering Convergence", fontweight="bold")
    fig.tight_layout()
    fig.savefig(ITER_DIR / "convergence_summary.pdf", bbox_inches="tight", dpi=150)
    fig.savefig(ITER_DIR / "convergence_summary.png", bbox_inches="tight", dpi=150)
    plt.show()

    with open(ITER_DIR / "round_summary.json", "w") as f:
        _json.dump(iter_results, f, indent=2)

    print(
        f"\n{'Round':<6} {'ACF@50':<8} {'ACF@50(nd)':<11} {'ESS':<8} {'ESS(nd)':<9} "
        f"{'#near1':<7} {'swap':<6} {'cond(M)':<10}"
    )
    print("-" * 80)
    for r in iter_results:
        print(
            f"{r['round']:<6} {r['acf50_median']:<8.3f} {r['acf50_nd_median']:<11.3f} "
            f"{r['ess_median']:<8.1f} {r['ess_nd_median']:<9.1f} "
            f"{r['n_HtM_near1']:<7} {r['swap_acceptance_rate']:<6.3f} "
            f"{r['cond_M_cold']:<10.2e}"
        )

    final = iter_results[-1]
    first = iter_results[0]
    gain = final["ess_median"] / max(first["ess_median"], 0.1)
    print(
        f"\n★ H3-1 Final: ACF@50={final['acf50_median']:.3f}  "
        f"ESS={final['ess_median']:.1f}  swap={final['swap_acceptance_rate']:.3f}  "
        f"gain={gain:.2f}×"
    )
    print(
        f"  Saved: {ITER_DIR / 'round_summary.json'}, convergence_summary.{{pdf,png}}"
    )


In [ ]:
# ── H3-1 vs H1-1 side-by-side ACF@50 / ESS comparison ──
if iter_results:
    h1_round_summary = (
        DATA_DIR / "results_no_hessian" / RUN_NAME / "iterative" / "round_summary.json"
    )
    if h1_round_summary.exists():
        with open(h1_round_summary) as f:
            h1_results = _json.load(f)

        h1_rounds = [r["round"] for r in h1_results]
        h1_acf50 = [r["acf50_median"] for r in h1_results]
        h1_ess = [r["ess_median"] for r in h1_results]

        fig, (axa, axb) = plt.subplots(1, 2, figsize=(14, 5))

        axa.plot(h1_rounds, h1_acf50, "o-", color="steelblue", lw=2, label="H1-1 (no Hessian init)")
        axa.plot(rounds, acf50s, "s-", color="crimson", lw=2, label="H3-1 (tempered)")
        axa.axhline(0.3, color="green", ls=":", lw=1, alpha=0.7)
        axa.axhline(0.5, color="red", ls=":", lw=1, alpha=0.7)
        axa.set_xlabel("Round")
        axa.set_ylabel("ACF@50 median (cold chain)")
        axa.set_title("ACF@50: H3-1 vs H1-1")
        axa.legend()
        axa.grid(True, alpha=0.3)
        axa.set_ylim(0, 1.05)

        axb.plot(h1_rounds, h1_ess, "o-", color="steelblue", lw=2, label="H1-1 (no Hessian init)")
        axb.plot(rounds, ess_meds, "s-", color="crimson", lw=2, label="H3-1 (tempered)")
        axb.set_xlabel("Round")
        axb.set_ylabel("ESS median (cold chain)")
        axb.set_title("ESS: H3-1 vs H1-1")
        axb.legend()
        axb.grid(True, alpha=0.3)

        fig.suptitle("Iterative convergence: tempering vs single-chain", fontweight="bold")
        fig.tight_layout()
        fig.savefig(ITER_DIR / "h3_vs_h1_convergence.pdf", bbox_inches="tight", dpi=150)
        fig.savefig(ITER_DIR / "h3_vs_h1_convergence.png", bbox_inches="tight", dpi=150)
        plt.show()
        print(f"✓ Saved H3-1 vs H1-1 comparison to {ITER_DIR / 'h3_vs_h1_convergence.pdf'}")
    else:
        print(f"H1-1 round_summary.json not found at {h1_round_summary}; skipping comparison.")


## 10. Cold-chain Mass Matrix Spectrum Evolution

Per H3-1 round, look at the **raw eigenspectrum of the cold-chain mass
matrix** — no Hessian product, no clipping. We just want to see how M
itself evolves across rounds.

1. **Heatmap** — round × eigenvalue rank (sorted descending per row),
   color = `log₁₀ eig(M_cold)`, sequential colormap, data-driven range.
2. **Per-rank trajectories** — eig(M_cold) at fixed sorted ranks across
   rounds (log y axis).


In [ ]:
# ── §10 prerequisites: load per-round M_cold; compute its eigenspectrum ──
# Requires §9 to have already run (`iter_round_paths` in scope).

if iter_round_paths:
    per_round_M_cold = []
    per_round_step_size = []
    for rp in iter_round_paths:
        rd = torch.load(rp, weights_only=False)
        per_round_M_cold.append(rd["per_chain_mass_matrix"][0])
        per_round_step_size.append(rd["per_chain_step_size"][0])

    rounds = [
        int(torch.load(rp, weights_only=False)["round"]) for rp in iter_round_paths
    ]
    n_rounds = len(rounds)
    d_dim = int(per_round_M_cold[0].shape[0]) if per_round_M_cold[0] is not None else 0

    # eig(M_cold) sorted descending per round
    M_sorted = torch.zeros(n_rounds, d_dim)
    for ri, M_r in enumerate(per_round_M_cold):
        if M_r is None:
            M_sorted[ri] = float("nan")
            continue
        eigvals_mr = torch.linalg.eigvalsh(M_r.detach().cpu().float())
        M_sorted[ri] = torch.sort(eigvals_mr, descending=True).values

    print(f"§10: built eig(M_cold) for {n_rounds} rounds × d={d_dim}")
else:
    print("No H3-1 rounds — skipping §10.")


In [ ]:
# ── §10a. Evolution heatmap: round × rank, log10 eig(M_cold), raw ──
# Round on x-axis (time flows left → right); eigenvalue rank on y-axis
# (rank 0 = largest eigenvalue, at the top). Color anchored to the global
# min/max of log10 eig(M_cold) so colors are directly comparable across rounds.
if iter_round_paths:
    log_M = torch.log10(M_sorted.clamp(min=1e-30))  # (n_rounds, d_dim)
    log_M_T = log_M.T  # (d_dim, n_rounds)

    finite_mask = torch.isfinite(log_M_T)
    vmin = float(log_M_T[finite_mask].min())
    vmax = float(log_M_T[finite_mask].max())

    fig, ax = plt.subplots(figsize=(max(6, 0.6 * n_rounds + 4), 7))
    im = ax.imshow(
        log_M_T.numpy(),
        aspect="auto",
        cmap="viridis",
        origin="upper",
        extent=[rounds[0] - 0.5, rounds[-1] + 0.5, d_dim, 1],
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel("Round")
    ax.set_ylabel("Eigenvalue rank (descending, 1 = largest)")
    ax.set_title(
        "Cold-chain mass matrix eigenspectrum evolution  (log₁₀ eig(M_cold))",
        fontweight="bold",
    )
    # Force integer ticks for rounds
    ax.set_xticks(rounds)
    plt.colorbar(im, ax=ax, label="log₁₀ eig(M_cold)")
    plt.tight_layout()
    fig.savefig(ITER_DIR / "m_spectrum_evolution.pdf", bbox_inches="tight", dpi=150)
    fig.savefig(ITER_DIR / "m_spectrum_evolution.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  range: log₁₀ eig ∈ [{vmin:.2f}, {vmax:.2f}]")


In [ ]:
# ── §10b. Per-rank trajectories: eig(M_cold) at fixed ranks across rounds ──
if iter_round_paths:
    fig, ax = plt.subplots(figsize=(10, 5))

    sample_ranks = sorted(
        set(
            r
            for r in [
                0,
                d_dim // 8,
                d_dim // 4,
                d_dim // 2,
                3 * d_dim // 4,
                7 * d_dim // 8,
                d_dim - 1,
            ]
            if 0 <= r < d_dim
        )
    )
    for r in sample_ranks:
        ax.plot(
            rounds,
            M_sorted[:, r].clamp(min=1e-30).numpy(),
            "o-",
            lw=1.5,
            label=f"rank {r}",
        )
    ax.set_yscale("log")
    ax.set_xlabel("H3-1 round")
    ax.set_ylabel("eig(M_cold) at fixed rank")
    ax.set_title("Cold-chain M eigenvalues per rank across rounds")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(ITER_DIR / "m_spectrum_trajectories.pdf", bbox_inches="tight", dpi=150)
    fig.savefig(ITER_DIR / "m_spectrum_trajectories.png", bbox_inches="tight", dpi=150)
    plt.show()
